# Last inn data

In [1]:
import pandas as pd

df = pd.read_json("lånekassen_data.json")
df["number_of_sents"] = df.fulltext.apply(lambda x: len([e for e in x if len(e)>2]))
df

,doc_hash,lang,url,domain,date,mimetype,fulltext,number_of_sents
0,b5fc6f81d5709f0051da1964c3900e7ca8df65f7,nno,http://lanekassen.no/globalassets/brosjyrer-fe...,lanekassen.no,2022-12-19 01:53:28,pdf,[Er du flyktning? Du blir rekna som flyktning ...,7
1,1fbe096fdfdd9916be1313006ac5cf44ac650231,nno,http://lanekassen.no/globalassets/skjemaer-fel...,lanekassen.no,2022-12-19 01:57:14,pdf,"[, , Du kan bruke dette skjemaet dersom du tar...",6
2,582aaf9a510b3bde21b5b3e13ffc3453d6ca8174,nno,http://lanekassen.no/globalassets/skjemaer-fel...,lanekassen.no,2022-12-19 01:57:21,pdf,"[, , Det er viktig at du les informasjonen på ...",8
3,32e23cf05e4db850a9e2f622c2edd0482572677e,nno,http://lanekassen.no/globalassets/skjemaer-fel...,lanekassen.no,2022-12-19 02:00:45,pdf,[Nynorsk Skjema for lærlinglønn Kor stort bort...,4
4,4ea46648c7ce59fe0c39b45779ed6402d8b41369,nno,http://lanekassen.no/globalassets/skjemaer-fel...,lanekassen.no,2022-12-19 02:01:01,pdf,[Nynorsk Skjema I – artikkelnr. 9001550 – nyno...,9
...,...,...,...,...,...,...,...,...
561,e29a43da7165d92d937c56416d50f563201ab5fa,nob,http://lanekassen.no/siteassets/skjemaer-og-fi...,lanekassen.no,2022-12-19 02:50:18,pdf,"[, , 01.01.2020 Storebrand Bank ASA 70 % Bolig...",4
562,c120ee7f582c758ae392b8f90795a7c664c39e90,nob,http://lanekassen.no/siteassets/skjemaer-og-fi...,lanekassen.no,2022-12-19 02:50:23,pdf,"[, , 06.11.2019 Storebrand Bank ASA 70 % Bolig...",6
563,553101a9b0a9fac935cf31e5dd59555d23ecbefe,nob,https://statistikk.lanekassen.no/globalassets/...,lanekassen.no,2022-12-19 03:10:41,pdf,"[, , Flyktningstipendet Mottakere av flyktning...",123
564,1f14c7bcc564613f79be7a8cae1a946e27cfd755,nob,https://statistikk.lanekassen.no/globalassets/...,lanekassen.no,2022-12-19 03:10:44,pdf,"[, , Tall og fakta om Lånekassens kunder og or...",24


In [2]:
assert len(set(df.doc_hash)) == len(df)

In [3]:
nynorske = df[df.lang == "nno"].copy()
nynorske.index = range(len(nynorske))

bokmålske = df[df.lang == "nob"].copy()
bokmålske.index = range(len(bokmålske))

len(nynorske), len(bokmålske)

(256, 310)

In [4]:
from collections import defaultdict

nynorske_sentences = defaultdict(list)
for t, df_ in nynorske.explode("fulltext").groupby("fulltext"):
    if len(t)>2:
        nynorske_sentences["text"].append(t)
        nynorske_sentences["doc_hashes"].append(set(df_.doc_hash))
        nynorske_sentences["urls"].append(set(df_.url))

nynorske_flat = pd.DataFrame(nynorske_sentences)

bokmålske_sentences = defaultdict(list)
for t, df_ in bokmålske.explode("fulltext").groupby("fulltext"):
    if len(t)>2:
        bokmålske_sentences["text"].append(t)
        bokmålske_sentences["doc_hashes"].append(set(df_.doc_hash))
        bokmålske_sentences["urls"].append(set(df_.url))

bokmålske_flat = pd.DataFrame(bokmålske_sentences)

In [5]:
nynorske_flat

,text,doc_hashes,urls
0,"""Alltid i jobb"" ""Mista/sagt opp jobben"" ""Komme...",{fe91c75291cdff5cc7a4358d1bf366b112736bdd},{https://statistikk.lanekassen.no/globalassets...
1,(01.01.2023–31.12.2025) (01.01.2023–31.12.2027...,{c8877c6d8b828ca9f33bebbabb17b8d3dcbf6c49},{http://lanekassen.no/siteassets/skjemaer-og-f...
2,(Felles studentsystem),{9b188ba61a4a422250cc1c1b305a34da86b4868b},{https://larestedsinfo.lanekassen.no/nn-NO/rap...
3,",må du først ha sendt inn ein søknad om lån og...",{0b405c19a11f64a0e8e81eacb621b06eb2befd4f},{http://lanekassen.no/globalassets/skjemaer-fe...
4,- Eg er stolt av arbeidsplassen min!,{6b7a9134b556e58c11f69f261db228dca22f4f75},{http://lanekassen.no/nn-NO/presse-og-samfunns...
...,...,...,...
4021,… med nysgjerrigheit og påverkingskraft,{1f428c88682b5026299321840d01f1a77a4f9fa9},{http://lanekassen.no/nn-NO/presse-og-samfunns...
4022,… over heile landet,{1f428c88682b5026299321840d01f1a77a4f9fa9},{http://lanekassen.no/nn-NO/presse-og-samfunns...
4023,… på ny teknologi,{1f428c88682b5026299321840d01f1a77a4f9fa9},{http://lanekassen.no/nn-NO/presse-og-samfunns...
4024,… samtidig som vi har det gøy og lærer,{1f428c88682b5026299321840d01f1a77a4f9fa9},{http://lanekassen.no/nn-NO/presse-og-samfunns...


In [6]:
len(set(nynorske_flat.doc_hashes.apply(tuple))), len(nynorske)

(371, 256)

# Bygg graf basert på setnigns-alignment

In [7]:
from pathlib import Path
from sentence_transformers import SentenceTransformer, util
import numpy as np

base_path = "/nb_sbert/texts_flat"

emb_path = Path(f"embeddings/{base_path}.npz")
if emb_path.exists():
    loaded = np.load(emb_path)
    bokmål_embeddings = loaded["a"]
    nynorsk_embeddings = loaded["b"]
else:
    model = SentenceTransformer('NbAiLab/nb-sbert-base', device="cuda")
    # basemodel_max_len = model[0].auto_model.config.max_position_embeddings
    # model.max_seq_length = basemodel_max_len
    emb_path.parent.mkdir(exist_ok=True, parents=True)
    nynorsk_embeddings = model.encode(nynorske_flat.text)
    bokmål_embeddings = model.encode(bokmålske_flat.text)
    np.savez_compressed(emb_path, a=bokmål_embeddings, b=nynorsk_embeddings)


search_result = util.semantic_search(nynorsk_embeddings, bokmål_embeddings, top_k=3)

## Vektet graf med score

In [8]:
import networkx as nx

G = nx.MultiDiGraph()

for e in nynorske_flat.itertuples():
    i = e.Index
    seen_doc_hashes = set()
    for res in search_result[i]:
        bm_doc_hashes = bokmålske_flat.iloc[res["corpus_id"]].doc_hashes
        score = res["score"]

        for nn_h in e.doc_hashes:
            for bm_h in bm_doc_hashes:
                if bm_h in seen_doc_hashes:
                    # only add one edge between two documents for each sentence
                    continue
                seen_doc_hashes.add(bm_h)
                G.add_edge(nn_h, bm_h, weight=score, no_bm_docs=len(bm_doc_hashes))

In [9]:
from collections import Counter, defaultdict

doc_matches = defaultdict(list)
threshold = 0.75

for e in nynorske.itertuples():
    node = e.doc_hash
    weights_per_node = defaultdict(list)
    for nn_node, bm_node, weight in G.edges(node, data="weight"):
        weights_per_node[bm_node].append(weight)
    for bm_node, scores in weights_per_node.items():
        weighted_mean_score = np.mean(scores) * (len(scores)/e.number_of_sents)
        if weighted_mean_score > threshold:
            doc_matches["nynorsk_doc_hash"].append(node)
            doc_matches["bokmål_doc_hash"].append(bm_node)
            doc_matches["weighted_mean_score"].append(weighted_mean_score)

matches = pd.DataFrame(doc_matches)
matches = matches.merge(nynorske[["doc_hash", "fulltext", "url"]], left_on="nynorsk_doc_hash", right_on="doc_hash").merge(bokmålske[["doc_hash", "fulltext", "url"]], left_on="bokmål_doc_hash", right_on="doc_hash", suffixes=["_nn", "_bm"])

matches

,nynorsk_doc_hash,bokmål_doc_hash,weighted_mean_score,doc_hash_nn,fulltext_nn,url_nn,doc_hash_bm,fulltext_bm,url_bm
0,b5fc6f81d5709f0051da1964c3900e7ca8df65f7,e0833f7f3c27e568e5cb83d20d461d46100aa636,0.985541,b5fc6f81d5709f0051da1964c3900e7ca8df65f7,[Er du flyktning? Du blir rekna som flyktning ...,http://lanekassen.no/globalassets/brosjyrer-fe...,e0833f7f3c27e568e5cb83d20d461d46100aa636,[Er du flyktning? Du regnes som flyktning hvis...,http://lanekassen.no/globalassets/brosjyrer-fe...
1,1fbe096fdfdd9916be1313006ac5cf44ac650231,aec569412f829b88a91775d3d629a807db50bc96,0.812720,1fbe096fdfdd9916be1313006ac5cf44ac650231,"[, , Du kan bruke dette skjemaet dersom du tar...",http://lanekassen.no/globalassets/skjemaer-fel...,aec569412f829b88a91775d3d629a807db50bc96,"[, , Du kan bruke dette skjemaet hvis du tar h...",http://lanekassen.no/globalassets/skjemaer-fel...
2,32e23cf05e4db850a9e2f622c2edd0482572677e,c74ba58b7ffbf1656e6bcb7da859d8fc73c33315,0.961291,32e23cf05e4db850a9e2f622c2edd0482572677e,[Nynorsk Skjema for lærlinglønn Kor stort bort...,http://lanekassen.no/globalassets/skjemaer-fel...,c74ba58b7ffbf1656e6bcb7da859d8fc73c33315,[Bokmål Skjema for lærlinglønn Hvor stort bort...,http://lanekassen.no/globalassets/skjemaer-fel...
3,4ea46648c7ce59fe0c39b45779ed6402d8b41369,f18361cc85817446b6846b541eff95e82cc82d8f,0.750898,4ea46648c7ce59fe0c39b45779ed6402d8b41369,[Nynorsk Skjema I – artikkelnr. 9001550 – nyno...,http://lanekassen.no/globalassets/skjemaer-fel...,f18361cc85817446b6846b541eff95e82cc82d8f,[Bokmål Skjema I – artikkelnr. 9001550 – bokmå...,http://lanekassen.no/globalassets/skjemaer-fel...
4,e5ffa18d5c02e8c1c756780ecd39ccad7a402229,ea5b53732f4a4656943a0acea89a25461df7d729,0.967385,e5ffa18d5c02e8c1c756780ecd39ccad7a402229,"[, , Du kan logge inn på og endre kontonummere...",http://lanekassen.no/globalassets/skjemaer-fel...,ea5b53732f4a4656943a0acea89a25461df7d729,"[, , Du kan logge inn på og endre kontonummere...",http://lanekassen.no/globalassets/skjemaer-fel...
...,...,...,...,...,...,...,...,...,...
184,339fb16e157307b49443bdb0ea160d07ba6627ef,4b23d053562d15cf1576cc15318dd3eb52fe5a52,0.969200,339fb16e157307b49443bdb0ea160d07ba6627ef,[Generelt om klargjering av søknader for vaksn...,https://larestedsinfo.lanekassen.no/nn-NO/rapp...,4b23d053562d15cf1576cc15318dd3eb52fe5a52,[Generelt om klargjøring av søknader for voksn...,https://larestedsinfo.lanekassen.no/nb-NO/rapp...
185,b8997c20c69c5ada72f0c1168781ced5f5940ecb,24db1028055031f34e1dae0c2bdabe5c3b8c5ed1,0.929908,b8997c20c69c5ada72f0c1168781ced5f5940ecb,[No kan elevar og studentar søke om straumstip...,https://larestedsinfo.lanekassen.no/nn-NO/nyhe...,24db1028055031f34e1dae0c2bdabe5c3b8c5ed1,[Nå kan elever og studenter søke om strømstipe...,https://larestedsinfo.lanekassen.no/nb-NO/nyhe...
186,125a109bcfd9662f6ff605a402c7042ab0fec8ca,aacb155a0fcc7887ca2d507a709a59bc8fff9e27,0.970364,125a109bcfd9662f6ff605a402c7042ab0fec8ca,"[Ekstra betalingsutsettingar ut 2022, Alle som...",https://larestedsinfo.lanekassen.no/nn-NO/nyhe...,aacb155a0fcc7887ca2d507a709a59bc8fff9e27,"[Ekstra betalingsutsettelser ut 2022, Alle som...",https://larestedsinfo.lanekassen.no/nb-NO/nyhe...
187,6e3448b22b4de4c8147b86ec4d003a68c05a34f5,1665d2bbe651b4090b9655dd4e06421578e039fc,0.976609,6e3448b22b4de4c8147b86ec4d003a68c05a34f5,[Elevar kan få omgjort delar av tilleggslån ko...,https://larestedsinfo.lanekassen.no/nn-NO/nyhe...,1665d2bbe651b4090b9655dd4e06421578e039fc,[Elever kan få omgjort deler av tilleggslån ko...,https://larestedsinfo.lanekassen.no/nb-NO/nyhe...


In [10]:
for e in matches.itertuples():
    if e.weighted_mean_score < 0.8:
        print(e.weighted_mean_score)
        print(e.fulltext_nn[:5])
        print("")
        print(e.fulltext_bm[:5])
        print("\n\n")

0.7508983612060547
['Nynorsk Skjema I – artikkelnr. 9001550 – nynorsk – 01.2022 – organisasjonsnummer NO 960 885 406 Namn, adresse og postnummer/-stad Personnummer Fødselsdato Kundenummer i Lånekassen Søknad om betalingsutsetting og/eller sletting av renter Du kan bruke nettsøknaden på', '', 'Kva månad/månader søker du om utsetting for? Du kan søke om utsetting for inntil seks månader fram i tid. Arbeidsløyse Lånekassen hentar informasjon om arbeidsløyse frå Nav. Unntak: Kvalifiseringsstønad i Noreg og registrert arbeidsløyse i utlandet må du dokumentere sjølv. Arbeidsavklaringspengar Lånekassen hentar informasjon om arbeidsavklaringspengar frå Nav. Arbeidsavklaringspengar, eller tilsvarande yting i utlandet, må du dokumentere sjølv. Sjukdom Legg ved kopi av sjukmelding eller legeattest for søknadsperioden. Fødsel eller adopsjon Legg ved kopi av fødselsattest dersom barnet ikkje er registrert i Folkeregisteret. Legg ved stadfesting på adopsjonsdato. Omsorg for pleietrengande i den nærm

## Sammenlikn med fasit

In [11]:
fasit = pd.read_csv("lanekassen_fasit.csv")
fasit[:10]

,nynorsk_doc_hash,bokmål_doc_hash,nynorsk_url,bokmål_url
0,87466e17316f039abfc03f4262f462c543be6857,9550f3c0398b025299b2c7760220854a3ad3a27e,http://lanekassen.no/nn-NO/stipend-og-lan/omgj...,http://lanekassen.no/nb-NO/stipend-og-lan/omgj...
1,4916b1fdf87274c1dd4e951ad0f01e7341f5505e,b1d60718e799422d237a785b6e04d2b89b8f03e3,http://lanekassen.no/nn-NO/stipend-og-lan/innt...,http://lanekassen.no/nb-NO/stipend-og-lan/innt...
2,7b19b48fcfa96b68d94b98ec6a2f5179622eacf8,56e51c6fb83b8f28350f7356a59e11432c495284,http://lanekassen.no/nn-NO/stipend-og-lan/fors...,http://lanekassen.no/nb-NO/stipend-og-lan/fors...
3,8a25a429e092b29bc807ef2ee2033c44a473fd44,b2179a3645fd40b4254a0d8cf6bdb0be2dff9596,http://lanekassen.no/nn-NO/stipend-og-lan/till...,http://lanekassen.no/nb-NO/stipend-og-lan/till...
4,51474766f1ed43201521d3e2ad140d0d8ced3493,526c9d7e11ada43275ba2a244a40a4432f3c232c,http://lanekassen.no/nn-NO/stipend-og-lan/nett...,http://lanekassen.no/nb-NO/stipend-og-lan/nett...
5,98528eae386cc4174b93b24a792c7c2934a36220,662a55a8b719dbf8eca41a3428d5aca935108645,http://lanekassen.no/nn-NO/stipend-og-lan/nord...,http://lanekassen.no/nb-NO/stipend-og-lan/nord...
6,317f12c236e7bfc308e12fad0d319f8ccd887bc6,44afe957f7f8a41921653cb9e122e65e785195d8,http://lanekassen.no/nn-NO/stipend-og-lan/nord...,http://lanekassen.no/nb-NO/stipend-og-lan/nord...
7,3347625331189ab43d9fe439da3f81e48eb08aa8,e36da3e23d23e07abbe3a95d0f7566adb40b183e,http://lanekassen.no/nn-NO/gjeld-og-betaling/s...,http://lanekassen.no/nb-NO/gjeld-og-betaling/s...
8,8aa70e5f2107df967ff7b68087e3a6aab062c7d9,37e3963e5abc44b3b5af1e806e17692cc4867e41,http://lanekassen.no/nn-NO/stipend-og-lan/nord...,http://lanekassen.no/nb-NO/stipend-og-lan/nord...
9,f5b7a2219eea1746a54e8ab5dd7274d191849648,8e3519c91614472b7a0b4f2af48cf39e2eebc934,http://lanekassen.no/nn-NO/stipend-og-lan/nord...,http://lanekassen.no/nb-NO/stipend-og-lan/nord...


In [12]:
matches[:10]

,nynorsk_doc_hash,bokmål_doc_hash,weighted_mean_score,doc_hash_nn,fulltext_nn,url_nn,doc_hash_bm,fulltext_bm,url_bm
0,b5fc6f81d5709f0051da1964c3900e7ca8df65f7,e0833f7f3c27e568e5cb83d20d461d46100aa636,0.985541,b5fc6f81d5709f0051da1964c3900e7ca8df65f7,[Er du flyktning? Du blir rekna som flyktning ...,http://lanekassen.no/globalassets/brosjyrer-fe...,e0833f7f3c27e568e5cb83d20d461d46100aa636,[Er du flyktning? Du regnes som flyktning hvis...,http://lanekassen.no/globalassets/brosjyrer-fe...
1,1fbe096fdfdd9916be1313006ac5cf44ac650231,aec569412f829b88a91775d3d629a807db50bc96,0.812720,1fbe096fdfdd9916be1313006ac5cf44ac650231,"[, , Du kan bruke dette skjemaet dersom du tar...",http://lanekassen.no/globalassets/skjemaer-fel...,aec569412f829b88a91775d3d629a807db50bc96,"[, , Du kan bruke dette skjemaet hvis du tar h...",http://lanekassen.no/globalassets/skjemaer-fel...
2,32e23cf05e4db850a9e2f622c2edd0482572677e,c74ba58b7ffbf1656e6bcb7da859d8fc73c33315,0.961291,32e23cf05e4db850a9e2f622c2edd0482572677e,[Nynorsk Skjema for lærlinglønn Kor stort bort...,http://lanekassen.no/globalassets/skjemaer-fel...,c74ba58b7ffbf1656e6bcb7da859d8fc73c33315,[Bokmål Skjema for lærlinglønn Hvor stort bort...,http://lanekassen.no/globalassets/skjemaer-fel...
3,4ea46648c7ce59fe0c39b45779ed6402d8b41369,f18361cc85817446b6846b541eff95e82cc82d8f,0.750898,4ea46648c7ce59fe0c39b45779ed6402d8b41369,[Nynorsk Skjema I – artikkelnr. 9001550 – nyno...,http://lanekassen.no/globalassets/skjemaer-fel...,f18361cc85817446b6846b541eff95e82cc82d8f,[Bokmål Skjema I – artikkelnr. 9001550 – bokmå...,http://lanekassen.no/globalassets/skjemaer-fel...
4,e5ffa18d5c02e8c1c756780ecd39ccad7a402229,ea5b53732f4a4656943a0acea89a25461df7d729,0.967385,e5ffa18d5c02e8c1c756780ecd39ccad7a402229,"[, , Du kan logge inn på og endre kontonummere...",http://lanekassen.no/globalassets/skjemaer-fel...,ea5b53732f4a4656943a0acea89a25461df7d729,"[, , Du kan logge inn på og endre kontonummere...",http://lanekassen.no/globalassets/skjemaer-fel...
5,d916f3684df7402700c3d90a4ddbe1f7a7908011,cbdb48a604c72e1336686a6e7b65af8ef1b39872,0.765474,d916f3684df7402700c3d90a4ddbe1f7a7908011,"[, , Kundenummer i Lånekassen Fødselsnummer (1...",http://lanekassen.no/globalassets/skjemaer-fel...,cbdb48a604c72e1336686a6e7b65af8ef1b39872,"[, , Kundenummer i Lånekassen Fødselsnummer (1...",http://lanekassen.no/globalassets/skjemaer-fel...
6,581f31cb211681f4d4a7649402b2745e8c76ab41,b3b6c37648b7f606148a513778adf4df9abcaeef,0.878825,581f31cb211681f4d4a7649402b2745e8c76ab41,[Fastsett av Kunnskapsdepartementet 25. februa...,http://lanekassen.no/siteassets/skjemaer-og-fi...,b3b6c37648b7f606148a513778adf4df9abcaeef,"[, , Fastsatt av Kunnskapsdepartementet 21. de...",http://lanekassen.no/siteassets/skjemaer-og-fi...
7,36829e447d39513bd58d3927408d5077c298f852,d5051725feffc9ce943032e97d5c37a53583ffa4,0.964679,36829e447d39513bd58d3927408d5077c298f852,[Du finn informasjon om vilkår og dokumentasjo...,http://lanekassen.no/globalassets/skjemaer-fel...,d5051725feffc9ce943032e97d5c37a53583ffa4,[Du finner informasjon om vilkår og dokumentas...,http://lanekassen.no/globalassets/skjemaer-fel...
8,87541c4a30cf3d7498472ecf5bd3b24ae3b54a64,1a325ea0310ea1a5ef1af7f83bce3079504599ec,0.870077,87541c4a30cf3d7498472ecf5bd3b24ae3b54a64,"[Søknaden fortset på neste side, , Tilleggssti...",http://lanekassen.no/globalassets/skjemaer-fel...,1a325ea0310ea1a5ef1af7f83bce3079504599ec,"[Søknaden fortsetter på neste side, , Tilleggs...",http://lanekassen.no/globalassets/skjemaer-fel...
9,0a96951cc5a70ccbff1865a5f500c0cac315150c,8f92ec11cc39893563836438cdf468b73d4fcb6d,0.885381,0a96951cc5a70ccbff1865a5f500c0cac315150c,[Sletting av gjeld for lærarar i grunnskolen N...,http://lanekassen.no/globalassets/skjemaer-fel...,8f92ec11cc39893563836438cdf468b73d4fcb6d,[Sletting av gjeld for lærere i grunnskolen Bo...,http://lanekassen.no/globalassets/skjemaer-fel...


In [13]:
treff = fasit[["nynorsk_doc_hash", "bokmål_doc_hash"]].merge(matches, on="nynorsk_doc_hash")
treff = treff[treff.bokmål_doc_hash_x == treff.bokmål_doc_hash_y][["weighted_mean_score", "doc_hash_nn", "fulltext_nn", "url_nn", "doc_hash_bm","fulltext_bm", "url_bm"]]

In [14]:
len(treff), len(fasit)

(43, 55)